In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
## Initialize the model
model=SentenceTransformer('all-MiniLM-L6-v2')

## Sample text
text="""
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""
sentences=[s.strip() for s in text.split("\n") if s.strip()]

### sstep 2: Embed each setence
embeddings=model.encode(sentences)

# Step 3: Initialize parameters
threshold = 0.7  # control chunk tightness
chunks = []
current_chunk=[sentences[0]]

## Step 4: Semantic grouping based on threshold

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i - 1]],
        [embeddings[i]]
    )[0][0]

    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]

# Append the last chunk
chunks.append(" ".join(current_chunk))

# Output the chunks
print("\n📌 Semantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}:\n{chunk}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


📌 Semantic Chunks:

Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
You can create chains, agents, memory, and retrievers.

Chunk 3:
The Eiffel Tower is located in Paris.

Chunk 4:
France is a popular tourist destination.


In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# LangChain core
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableMap

# LangChain community
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI


C:\Users\Adit Sharma\AppData\Local\Temp\ipykernel_10280\2431187682.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [5]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("groq_api")

In [6]:
### Custom Semantic Chunker With Threshold

class ThresholdSematicChunker:
    def __init__(self,model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model=SentenceTransformer(model_name)
        self.threshold=threshold 

    def split(self, text: str):
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]

        chunks.append(". ".join(current_chunk) + ".")
        return chunks
    
    def split_documents(self,docs):
        result=[]
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))

        return result

    

In [7]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [8]:
chunker=ThresholdSematicChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [16]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("groq_api")

# Step 1: Use HuggingFace embeddings (instead of OpenAI)
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Step 2: Build FAISS vector store
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever()

# Step 3: Initialize Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"]
)

# Step 4: Query pipeline
query = "Explain cosine similarity in embeddings"
docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in docs])

response = llm.invoke(f"Context:\n{context}\n\nQuestion: {query}")
print(response)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

content='### Cosine Similarity – The “Angle” Between Embeddings\n\nIn modern LLM‑powered applications (e.g., LangChain, retrieval‑augmented generation), *embeddings* are dense vectors that encode the semantic meaning of text, images, or other modalities.  \nOnce you have two embeddings, you need a way to say “how similar are they?”  \nCosine similarity is the most common answer.\n\n---\n\n## 1. What is it?\n\nCosine similarity measures the cosine of the angle between two vectors in a high‑dimensional space.\n\n- **Range**:  \n  -1  → 180° (opposite directions)  \n   0  → 90°  (orthogonal, no similarity)  \n   1  → 0°   (identical direction)\n\nBecause embeddings from language models are usually non‑negative and normalized, the practical range is **0\u202f→\u202f1**.\n\n---\n\n## 2. The Formula\n\nFor two vectors **A** and **B**:\n\n\\[\n\\text{cos\\_sim}(A,B) = \n\\frac{A \\cdot B}{\\|A\\|\\;\\|B\\|} \n= \\frac{\\sum_{i=1}^{n} A_i B_i}{\\sqrt{\\sum_{i=1}^{n} A_i^2}\\;\\sqrt{\\sum_{i=1}

In [17]:
template = """Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [22]:
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableMap
from langchain_core.output_parsers import StrOutputParser

# --- 1. Initialize Groq LLM ---
llm = ChatGroq(
    model="openai/gpt-oss-20b",   
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.4
)

# --- 2. LCEL Chain with retrieval ---
rag_chain = (
    RunnableMap(
        {
            "context": lambda x: retriever.invoke(x["question"]),
            "question": lambda x: x["question"],
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

# --- 3. Run Query ---
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)


LangChain is a framework that lets you build applications powered by large language models (LLMs). It offers modular abstractions for combining LLMs with external tools (such as OpenAI, Pinecone, etc.) and for creating components like chains, agents, memory modules, and retrievers.


In [24]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

In [27]:
# HuggingFace embeddings (no API key needed)
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create semantic chunker
chunker = SemanticChunker(embedding)

# Load documents
loader = TextLoader("langchain_intro.txt")
docs = loader.load()

# Split into semantic chunks
chunks = chunker.split_documents(docs)

print(f"Number of chunks: {len(chunks)}")
print(chunks[0].page_content)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of chunks: 2
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.
